In [2]:
#Selection, Interpolation and Slicing
from datetime import timedelta
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


In [4]:
# temperature data
rand_data = 283 + 5 * np.random.randn(5, 3, 4)
times_index = pd.date_range("2018-01-01", periods=5)
lons = np.linspace(-120, -60, 4)
lats = np.linspace(25, 55, 3)
temperature = xr.DataArray(
    rand_data, coords=[times_index, lats, lons], dims=["time", "lat", "lon"]
)
temperature.attrs["units"] = "Kelvin"
temperature.attrs["standard_name"] = "air_temperature"

# pressure data
pressure_data = 1000.0 + 5 * np.random.randn(5, 3, 4)
pressure = xr.DataArray(
    pressure_data, coords=[times_index, lats, lons], dims=["time", "lat", "lon"]
)
pressure.attrs["units"] = "hPa"
pressure.attrs["standard_name"] = "air_pressure"

# combinate temperature and pressure DataArrays into a Dataset called 'ds'
ds = xr.Dataset(data_vars={"Temperature": temperature, "Pressure": pressure})
ds

<xarray.Dataset> Size: 1kB
Dimensions:      (time: 5, lat: 3, lon: 4)
Coordinates:
  * time         (time) datetime64[us] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat          (lat) float64 24B 25.0 40.0 55.0
  * lon          (lon) float64 32B -120.0 -100.0 -80.0 -60.0
Data variables:
    Temperature  (time, lat, lon) float64 480B 285.9 285.3 278.8 ... 280.7 286.6
    Pressure     (time, lat, lon) float64 480B 999.3 1.001e+03 ... 997.8 998.1

In [6]:
#NumPy-like Selection
indexed_selection = temperature[
    1,:,:
]
indexed_selection

<xarray.DataArray (lat: 3, lon: 4)> Size: 96B
array([[281.86426231, 289.17118607, 286.42769501, 288.83127738],
       [284.81090528, 283.28854756, 276.66576001, 286.83306715],
       [276.76413408, 282.34985476, 277.0357315 , 277.06952511]])
Coordinates:
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
    time     datetime64[us] 8B 2018-01-02
Attributes:
    units:          Kelvin
    standard_name:  air_temperature

In [8]:
#Section  .sel
named_selection = temperature.sel(time="2018-01-02")
named_selection

<xarray.DataArray (lat: 3, lon: 4)> Size: 96B
array([[281.86426231, 289.17118607, 286.42769501, 288.83127738],
       [284.81090528, 283.28854756, 276.66576001, 286.83306715],
       [276.76413408, 282.34985476, 277.0357315 , 277.06952511]])
Coordinates:
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
    time     datetime64[us] 8B 2018-01-02
Attributes:
    units:          Kelvin
    standard_name:  air_temperature

In [12]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', '_i2', 'timedelta', 'np', 'pd', 'xr', 'plt', '_i3', 'rand_data', 'time_index', 'lons', 'lats', '_i4', 'times_index', 'temperature', 'pressure_data', 'pressure', 'ds', '_4', '_i5', '_i6', 'indexed_selection', '_6', '_i7', 'name_selection', '_i8', 'named_selection', '_8', '_i9', '_i10', '_i11', '_i12'])


In [13]:
named_selection = named_selection.sel(
    lat=25,
    lon=-120,
    method="nearest"
)

In [14]:
print(ds)

<xarray.Dataset> Size: 1kB
Dimensions:      (time: 5, lat: 3, lon: 4)
Coordinates:
  * time         (time) datetime64[us] 40B 2018-01-01 2018-01-02 ... 2018-01-05
  * lat          (lat) float64 24B 25.0 40.0 55.0
  * lon          (lon) float64 32B -120.0 -100.0 -80.0 -60.0
Data variables:
    Temperature  (time, lat, lon) float64 480B 285.9 285.3 278.8 ... 280.7 286.6
    Pressure     (time, lat, lon) float64 480B 999.3 1.001e+03 ... 997.8 998.1


In [16]:
print(named_selection)

<xarray.DataArray ()> Size: 8B
array(281.86426231)
Coordinates:
    time     datetime64[us] 8B 2018-01-02
    lat      float64 8B 25.0
    lon      float64 8B -120.0
Attributes:
    units:          Kelvin
    standard_name:  air_temperature


In [17]:
coord_selection = ds.sel(
    lat=25,
    lon=-120,
    method="nearest"
)

coord_selection

<xarray.Dataset> Size: 136B
Dimensions:      (time: 5)
Coordinates:
  * time         (time) datetime64[us] 40B 2018-01-01 2018-01-02 ... 2018-01-05
    lat          float64 8B 25.0
    lon          float64 8B -120.0
Data variables:
    Temperature  (time) float64 40B 285.9 281.9 282.9 284.2 284.3
    Pressure     (time) float64 40B 999.3 998.2 1.001e+03 998.1 1e+03

In [18]:
#Nearest-neighbor Sampling
temperature.sel(
    time="2018-01-01",method="nearest",tolerance=timedelta(days=2)
)

<xarray.DataArray (lat: 3, lon: 4)> Size: 96B
array([[285.9178665 , 285.2990282 , 278.80336874, 280.63095707],
       [290.68957767, 284.62372906, 283.27680498, 282.96880836],
       [286.94295628, 288.8801176 , 282.72094513, 284.91095914]])
Coordinates:
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
    time     datetime64[us] 8B 2018-01-01
Attributes:
    units:          Kelvin
    standard_name:  air_temperature

In [24]:
#Interpolation
print(temperature.dims)

('time', 'lat', 'lon')


In [31]:
pip install scipy

   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
    --------------------------------------- 0.5/36.6 MB 3.4 MB/s eta 0:00:11
    --------------------------------------- 0.8/36.6 MB 2.4 MB/s eta 0:00:15
   - -------------------------------------- 1.3/36.6 MB 2.1 MB/s eta 0:00:17
   -- ------------------------------------- 1.8/36.6 MB 2.2 MB/s eta 0:00:16
   -- ------------------------------------- 1.8/36.6 MB 2.2 MB/s eta 0:00:16
   -- ------------------------------------- 2.4/36.6 MB 1.8 MB/s eta 0:00:19
   --- ------------------------------------ 2.9/36.6 MB 2.0 MB/s eta 0:00:17
   --- ------------------------------------ 3.4/36.6 MB 2.1 MB/s eta 0:00:16
   ---- ----------------------------------- 4.2/36.6 MB 2.3 MB/s eta 0:00:15
   ----- ---------------------------------- 4.7/36.6 MB 2.3 MB/s eta 0:00:14
   ------ --------------------------------- 5.5/36.6 MB 2.4 MB/s eta 0:00:13
   ------ --------------------------------- 6.0/36.6 MB 2.5 MB/s eta 0:00:13
   ---

In [32]:
import scipy
interp_temp = temperature.interp(
    lon=-105,
    lat=40,
    method="linear"
)

print(interp_temp)

<xarray.DataArray (time: 5)> Size: 40B
array([286.14019121, 283.66913699, 279.11715118, 282.95388623,
       286.51788846])
Coordinates:
  * time     (time) datetime64[us] 40B 2018-01-01 2018-01-02 ... 2018-01-05
    lon      int64 8B -105
    lat      int64 8B 40
Attributes:
    units:          Kelvin
    standard_name:  air_temperature


In [ ]:
#slicing Along Coordinates
temperature.sel(
    time= slice("2018-01-01","2018-01-03"),lon=slice(-110,-70),lat=slice(25,45)
)

<xarray.DataArray (time: 3, lat: 2, lon: 2)> Size: 96B
array([[[285.2990282 , 278.80336874],
        [284.62372906, 283.27680498]],

       [[289.17118607, 286.42769501],
        [283.28854756, 276.66576001]],

       [[280.00339108, 268.73414558],
        [279.83528405, 276.95634308]]])
Coordinates:
  * time     (time) datetime64[us] 24B 2018-01-01 2018-01-02 2018-01-03
  * lat      (lat) float64 16B 25.0 40.0
  * lon      (lon) float64 16B -100.0 -80.0
Attributes:
    units:          Kelvin
    standard_name:  air_temperature

In [34]:
#One More Selection Method: .loc
temperature.loc['2018-01-01']

<xarray.DataArray (lat: 3, lon: 4)> Size: 96B
array([[285.9178665 , 285.2990282 , 278.80336874, 280.63095707],
       [290.68957767, 284.62372906, 283.27680498, 282.96880836],
       [286.94295628, 288.8801176 , 282.72094513, 284.91095914]])
Coordinates:
  * lat      (lat) float64 24B 25.0 40.0 55.0
  * lon      (lon) float64 32B -120.0 -100.0 -80.0 -60.0
    time     datetime64[us] 8B 2018-01-01
Attributes:
    units:          Kelvin
    standard_name:  air_temperature

In [35]:
temperature.loc['2018-01-01':'2018-01-02',25:25,-110:-70]

<xarray.DataArray (time: 2, lat: 1, lon: 2)> Size: 32B
array([[[285.2990282 , 278.80336874]],

       [[289.17118607, 286.42769501]]])
Coordinates:
  * time     (time) datetime64[us] 16B 2018-01-01 2018-01-02
  * lat      (lat) float64 8B 25.0
  * lon      (lon) float64 16B -100.0 -80.0
Attributes:
    units:          Kelvin
    standard_name:  air_temperature

In [39]:
temperature.loc['2018-01-01':'2018-01-03', slice(25,45),-110:-70]

<xarray.DataArray (time: 3, lat: 2, lon: 2)> Size: 96B
array([[[285.2990282 , 278.80336874],
        [284.62372906, 283.27680498]],

       [[289.17118607, 286.42769501],
        [283.28854756, 276.66576001]],

       [[280.00339108, 268.73414558],
        [279.83528405, 276.95634308]]])
Coordinates:
  * time     (time) datetime64[us] 24B 2018-01-01 2018-01-02 2018-01-03
  * lat      (lat) float64 16B 25.0 40.0
  * lon      (lon) float64 16B -100.0 -80.0
Attributes:
    units:          Kelvin
    standard_name:  air_temperature